In [3]:
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

import matplotlib.pyplot as plt

    @staticmethod
    def train_department_decision_tree(conn, max_depth=4, test_size=0.3):
        """
        Train a decision tree using crew data to predict department_name.
        Features used:
            - age
            - sex
            - citizenship
            - adult_minor

        Returns a dictionary containing the model and encoders.
        """

        query = """
        SELECT
            c.age,
            c.sex,
            c.citizenship,
            c.adult_minor,
            d.department_name
        FROM crew c
        JOIN department d
            ON c.department_id = d.department_id
        WHERE c.age IS NOT NULL
          AND c.sex IS NOT NULL
          AND c.citizenship IS NOT NULL
          AND c.adult_minor IS NOT NULL
          AND d.department_name IS NOT NULL;
        """

        df = pd.read_sql_query(query, conn)

        if df.empty:
            print("No crew data found for training.")
            return None

        # Clean age column
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        df = df.dropna(subset=["age"])

        if df.empty:
            print("No valid numeric age values found after cleaning.")
            return None

        # Encode categorical fields
        sex_encoder = LabelEncoder()
        citizenship_encoder = LabelEncoder()
        adult_minor_encoder = LabelEncoder()
        target_encoder = LabelEncoder()

        df["sex_encoded"] = sex_encoder.fit_transform(df["sex"].astype(str))
        df["citizenship_encoded"] = citizenship_encoder.fit_transform(df["citizenship"].astype(str))
        df["adult_minor_encoded"] = adult_minor_encoder.fit_transform(df["adult_minor"].astype(str))
        df["target"] = target_encoder.fit_transform(df["department_name"].astype(str))

        # Features and target
        X = df[["age", "sex_encoded", "citizenship_encoded", "adult_minor_encoded"]]
        y = df["target"]

        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=42,
            stratify=y
        )

        # Train model
        model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)

        # Report
        print("\n==============================")
        print("DECISION TREE RESULTS")
        print("==============================")
        print("Accuracy:", accuracy_score(y_test, y_pred))

        print("\nClassification Report:\n")
        print(classification_report(
            y_test,
            y_pred,
            target_names=target_encoder.classes_,
            zero_division=0
        ))

        print("\nDecision Tree Rules:\n")
        print(export_text(
            model,
            feature_names=["age", "sex_encoded", "citizenship_encoded", "adult_minor_encoded"]
        ))

        print("\nEncoding Keys:")
        print("sex:", dict(zip(sex_encoder.classes_, sex_encoder.transform(sex_encoder.classes_))))
        print("citizenship:", dict(zip(citizenship_encoder.classes_, citizenship_encoder.transform(citizenship_encoder.classes_))))
        print("adult_minor:", dict(zip(adult_minor_encoder.classes_, adult_minor_encoder.transform(adult_minor_encoder.classes_))))
        print("department_name:", dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_))))

        # Plot tree
        plt.figure(figsize=(20, 10))
        plot_tree(
            model,
            feature_names=["age", "sex_encoded", "citizenship_encoded", "adult_minor_encoded"],
            class_names=target_encoder.classes_,
            filled=True,
            rounded=True,
            fontsize=9
        )
        plt.title("Decision Tree for Predicting Crew Department")
        plt.tight_layout()
        plt.show()

        return {
            "model": model,
            "target_encoder": target_encoder,
            "sex_encoder": sex_encoder,
            "citizenship_encoder": citizenship_encoder,
            "adult_minor_encoder": adult_minor_encoder,
            "training_dataframe": df
        }

IndentationError: unexpected indent (837826087.py, line 11)